In [191]:
# Checking Python executable
import sys
print(sys.executable) # MAKE SURE THIS POINTS TO THE CORRECT VIRTUAL ENVIRONMENT PATH FOR CORRECT PACKAGE INSTALLATION

/home/louis/miniconda3/envs/aml_lab/bin/python


In [192]:
# ALWAYS INSTALL USING %pip, NOT !pip (can sometimes install to system Python) or pip
# %pip install numpy
# %pip install pandas
# %pip install matplotlib
# %pip install scikit-learn
# %pip install torch # Using version 2.10.0+cu128
# %pip install torchinfo

In [193]:
# Import packages
import glob
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import torchinfo
print(torch.__version__)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu') # Define device (either GPU or CPU if GPU is unavailable)
# DEVICE = "cpu"
print(DEVICE)

##### Model training function #####
def train(
        model: nn.Module,
        train_loader: DataLoader,
        criterion: nn.Module,
        optimizer: torch.optim.Optimizer,
        num_epochs: int = 10,
        val_loader: DataLoader = None,
        device: torch.device = DEVICE,
        print_loss: bool = True, # Flag for whether to print loss outputs or not
):
    model = model.to(device) # Move the model to same device as data (GPU or CPU)

    # Train for the number of epochs specified
    for epoch in range(num_epochs):

        ### TRAINING SET ###
        model.train() # Set model to training mode (affects Dropout/BatchNorm)
        train_loss = 0.0 # Initialise training loss

        # Loop through all batches in the training set
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device) # Move data to same device as model (GPU or CPU)
            optimizer.zero_grad() # Clear gradients
            preds = model(inputs) # Forward pass, obtain predictions
            loss = criterion(preds, labels) # Compute loss based on predictions and true labels (loss is MEAN loss over the batch)
            loss.backward() # Backward pass, compute gradient of loss w.r.t every model parameter
            optimizer.step() # Update weights, using optimisation algorithm chosen

            train_loss += loss.item() * inputs.size(0) # Sum training loss of EACH SAMPLE in the batch (inputs.size(0) is batch size)
        train_loss /= len(train_loader.dataset) # Calculate mean loss PER SAMPLE over ENTIRE DATASET

        ### VALIDATION SET ###
        # Loop through all validation batches (if validation data is given)
        if val_loader is not None:
            model.eval() # Set model to evaluation (inference) mode (turns dropout OFF, and affects BatchNorm)
            val_loss = 0.0 # Initialise validation loss
            all_preds_class = [] # Initialise list to store output prediction classes
            all_labels = [] # Initialise list to store actual labels of output predictions

            with torch.no_grad(): # Disable gradient computing
                # Loop through all batches in the validation set
                for inputs, labels in val_loader:
                    inputs, labels = inputs.to(device), labels.to(device) # Move data to same device as model (GPU or CPU)
                    preds = model(inputs) # Forward pass, obtain predictions as LOGITS (NO FOLLOWING BACKWARD PASS IN VALIDATION)

                    # Compute confusion matrix values
                    preds_class = torch.argmax(preds, dim=1) # Get class index of logit predictions
                    all_preds_class.append(preds_class.cpu())
                    all_labels.append(labels.cpu())

                    # Compute validation loss
                    loss = criterion(preds, labels) # Compute loss based on predictions and true labels (loss is MEAN loss over the batch)
                    val_loss += loss.item() * inputs.size(0) # Sum validation loss of EACH SAMPLE in the batch (inputs.size(0) is batch size)
                val_loss /= len(val_loader.dataset)

                ### COMPUTE AND DISPLAY METRICS ###
                # Compute and display confusion matrix values
                all_preds_class = torch.cat(all_preds_class)
                all_labels = torch.cat(all_labels)

                cm = confusion_matrix(all_labels, all_preds_class)
                print("Validation confusion matrix:\n", cm)

                # Compute and display accuracy values
                label_names = ["Baseline", "Distracted", "Focused", "Stressed"]
                per_label_acc = cm.diagonal() / cm.sum(axis=1) * 100 # Correct predictions per label class (diag vals) / Total true samples per label class
                for l, acc in enumerate(per_label_acc):
                    print(f"Label: {label_names[l]}, Accuracy: {acc:.4f}%")

        ### PRINT TRAINING/VALIDATION OUTPUTS ###
            if print_loss:
                print(f"Epoch[{epoch+1}/{num_epochs}] Training Loss: {train_loss:.5f}, Validation Loss: {val_loss:.5f}")
        else:
            if print_loss:
                print(f"Epoch[{epoch+1}/{num_epochs}] Training Loss: {train_loss:.5f}")


##### Model evaluation function #####
def eval(
        model: nn.Module,
        test_loader: DataLoader,
        criterion: nn.Module,
        device: torch.device = DEVICE,
        print_loss: bool = True, # Flag for whether to print loss outputs or not
):
    model.eval() # Set model to evaluation mode
    test_loss = 0.0 # Initialise test loss
    all_preds_class = [] # Initialise list to store output prediction classes
    all_labels = [] # Initialise list to store actual labels of output predictions

    with torch.no_grad(): # Disable gradient computing
        # Loop through all batches in the test set
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device) # Move data to same device as model (GPU or CPU)
            preds = model(inputs) # Forward pass, obtain predictions as LOGITS (NO FOLLOWING BACKWARD PASS IN TESTING)

            # Compute confusion matrix values
            preds_class = torch.argmax(preds, dim=1) # Get class index of logit predictions
            all_preds_class.append(preds_class.cpu())
            all_labels.append(labels.cpu())
            
            # Compute validation loss
            loss = criterion(preds, labels) # Compute loss based on predictions and true labels (loss is MEAN loss over the batch)

            test_loss += loss.item() * inputs.size(0) # Sum test loss of EACH SAMPLE in the batch (inputs.size(0) is batch size)
        test_loss /= len(test_loader.dataset)

        ### COMPUTE AND DISPLAY METRICS ###
        # Compute and display confusion matrix values
        all_preds_class = torch.cat(all_preds_class)
        all_labels = torch.cat(all_labels)

        cm = confusion_matrix(all_labels, all_preds_class)
        print("Testing confusion matrix:\n", cm)

        # Compute and display accuracy values
        label_names = ["Baseline", "Distracted", "Focused", "Stressed"]
        per_label_acc = cm.diagonal() / cm.sum(axis=1) * 100 # Correct predictions per label class (diag vals) / Total true samples per label class
        for l, acc in enumerate(per_label_acc):
            print(f"Label: {label_names[l]}, Accuracy: {acc:.4f}%")

    if print_loss:
        print(f"Test Loss: {test_loss:.5f}")

2.10.0+cu128
cuda


In [194]:
### Import data ###
# Remove first two columns (they are just host_time and time)
# PREVIOUS VERSION 1 #
# data1 = pd.read_csv("Data/adi_focused.csv")
# data2 = pd.read_csv("Data/adi_stressed.csv")
# data3 = pd.read_csv("Data/louis_focused.csv")
# data4 = pd.read_csv("Data/louis_stressed.csv")

# PREVIOUS VERSION 2 #
# base_data1_raw = pd.read_csv("Data/adi_7_5min_baseline.csv").iloc[:, 2:]
# base_data2_raw = pd.read_csv("Data/emmanuel_7_5min_baseline.csv").iloc[:, 2:]
# base_data3_raw = pd.read_csv("Data/louis_7_5min_baseline.csv").iloc[:, 2:]

# dist_data1_raw = pd.read_csv("Data/adi_7_5min_distraction.csv").iloc[:, 2:]
# dist_data2_raw = pd.read_csv("Data/emmanuel_7_5min_diistraction.csv").iloc[:, 2:]
# dist_data3_raw = pd.read_csv("Data/louis_7_5min_distract.csv").iloc[:, 2:]

# foc_data1_raw = pd.read_csv("Data/adi_7_5min_focus.csv").iloc[:, 2:]
# foc_data2_raw = pd.read_csv("Data/emmanuel_7_5min_focus.csv").iloc[:, 2:]
# foc_data3_raw = pd.read_csv("Data/louis_7_5min_focus.csv").iloc[:, 2:]

# str_data1_raw = pd.read_csv("Data/adi_7_5min_stress.csv").iloc[:, 2:]
# str_data2_raw = pd.read_csv("Data/emmanuel_7_5min_stress.csv").iloc[:, 2:]
# str_data3_raw = pd.read_csv("Data/louis_7_5min_stress.csv").iloc[:, 2:]

# Initialisations
data_folder_path = Path("Data/") # Set data folder path
base_data_raw = [] # List of BASELINE dataframes
dist_data_raw = [] # List of DISTRACTED dataframes
foc_data_raw = [] # List of FOCUS dataframes
str_data_raw = [] # List of STRESSED dataframes

# Loop through all CSV files in the data folder
for file in data_folder_path.glob("*.csv"):
    df = pd.read_csv(file).iloc[:, 2:] # Read data of current file into a pandas dataframe
    filename = file.name.lower() # Lowercase name of file (in case)

    # Store dataframe into appropriate list based on what type of emotion it corresponds to (mentioned in the filename)
    if "baseline" in filename:
        base_data_raw.append(df)
    elif "distract" in filename:
        dist_data_raw.append(df)
    elif "focus" in filename:
        foc_data_raw.append(df)
    elif "stress" in filename:
        str_data_raw.append(df)
    else:
        print("UNKNOWN DATA FILE")

# Show number of datafiles in each list (DEBUGGING)
print(f"Baseline data files: {len(base_data_raw)}")
print(f"Distracted data files: {len(dist_data_raw)}")
print(f"Focused data files: {len(foc_data_raw)}")
print(f"Stressed data files: {len(str_data_raw)}")


# DEBUGGING
# base_data1.head()

Baseline data files: 7
Distracted data files: 7
Focused data files: 7
Stressed data files: 7


In [195]:
##### Process data #####
### Function for normalising data ###
def normalise_data(data_in): # INPUT DATA IS A PANDAS DATAFRAME
    return (data_in - data_in.mean()) / data_in.std()

### Function for obtaining training, validation and test splits
def split_data(data_in, train_split, val_split, test_split): # data_in is a list of pandas dataframes
    # Create empty pandas dataframes for storing training, validation and test data splits
    train_data = pd.DataFrame()
    val_data = pd.DataFrame()
    test_data = pd.DataFrame()

    # Loop through all datasets in the input list of datasets
    for dataset in data_in:
        # Obtain number of training, validation and test data
        train_num = round(dataset.shape[0]*train_split)
        val_num = round(dataset.shape[0]*val_split)
        test_num = dataset.shape[0] - train_num - val_num

        # Obtain the training, validation and test data splits
        train_data_curr = dataset.iloc[0:train_num]
        val_data_curr = dataset.iloc[train_num:train_num+val_num]
        test_data_curr = dataset.iloc[train_num+val_num:]

        # Concatenate current dataset's training, validation and test data to the overall data
        train_data = pd.concat([train_data, train_data_curr], axis=0, ignore_index=True)
        val_data = pd.concat([val_data, val_data_curr], axis=0, ignore_index=True)
        test_data = pd.concat([test_data, test_data_curr], axis=0, ignore_index=True)
    
    # Convert pandas dataframes to numpy arrays
    train_data_np = train_data.values.astype("float32")
    val_data_np = val_data.values.astype("float32")
    test_data_np = test_data.values.astype("float32")

    return train_data_np, val_data_np, test_data_np

### Function for obtaining array of class labels for a dataset
def create_labels(data_in, label):
    all_labels = [] # Initialise

    # Loop through all training, validation and test datasets
    for dataset in data_in:
        dataset_labels = np.ones(len(dataset), dtype=np.int64)*label # Create an array of desired labels for the current dataset
        all_labels.append(dataset_labels) # Append labels to output

    return all_labels[0], all_labels[1], all_labels[2] # Return array of labels for training, validation and test sets

### Function for obtaining windowed input-label pairs FOR A SINGLE DATASET (needed as our data is highly dependent on previous data) ###
# E.g. [x0, x1, x2] -> y       [x1, x2, x3] -> y ...
def window_single_data(dataset, labels, win_size=10, num_overlap = 0):
    # OLD CODE FOR SINGLE TIMESTEP DELAY OVERLAP
    # # print(range(len(dataset) - win_size))
    # # print(dataset)
    # input_seq = np.array([dataset[i:i+win_size, :] for i in range(len(dataset) - win_size)]) # Get input sequence with length = window length
    # seq_label = [labels[i+win_size, 0] for i in range(len(dataset) - win_size)] # Get corresponding output for each input window sequence

    # Determine step size (number of samples to skip) for sampling data depending on desired number of overlapping samples
    step = win_size - num_overlap
    
    input_seq = np.array([dataset[i:i+win_size, :] for i in range(0, len(dataset) - win_size, step)]) # Get input sequence with length = window length
    seq_label = [labels[i+win_size, 0] for i in range(0, len(dataset) - win_size, step)] # Get corresponding output for each input window sequence

    # print(input_seq)
    return np.array(input_seq), np.array(seq_label)

### Function for obtaining windowed input-label pairs FOR A LIST OF DATASETS ###
def window_data(data_in, labels_in, win_size=10, win_overlap=0):
    # Initialisations
    all_data = []
    all_labels = []

    # Loop through all datasets (in the data_in list)
    for dataset, labels in zip(data_in, labels_in):
        if len(dataset) == 0: # IF dataset is empty (when split is set to 0)
            continue # Skip empty datasets
        windowed_dataset, windowed_labels = window_single_data(dataset, np.vstack(labels), win_size, win_overlap) # Get windowed input-label pairs for current dataset
        all_data.append(windowed_dataset) # Store current windowed dataset
        all_labels.append(windowed_labels) # Store current windowed labels

    # if len(all_data) == 0:
    #     return np.array([]), np.array([])
    
    data_out = np.concatenate(all_data, axis=0) # Concatenate all windowed dataset
    labels_out = np.concatenate(all_labels, axis=0) # Concatenate all windowed labels

    return data_out, labels_out


# Initialisations
num_sensor_readings = 20 # Number of sensor readings
num_classes = 4 # Number of classification classes (number of emotional states to identify)
window_size = 16
window_overlap = 5 # Number of samples to overlap between windows (setting to window_size-1 gives a single timestep delay between windows)

base_data = [] # List of normalised BASELINE dataframes
dist_data = [] # List of normalised DISTRACTED dataframes
foc_data = [] # List of normalised FOCUSED dataframes
str_data = [] # List of normalised STRESSED dataframes

## Normalise all data ##
# Loop through all BASELINE dataframes
for df in base_data_raw:
    norm_df = normalise_data(df) # Normalise current dataframe
    base_data.append(norm_df) # Store normalised dataframe into list

# Loop through all DISTRACTED dataframes
for df in dist_data_raw:
    norm_df = normalise_data(df) # Normalise current dataframe
    dist_data.append(norm_df) # Store normalised dataframe into list

# Loop through all FOCUSED dataframes
for df in foc_data_raw:
    norm_df = normalise_data(df) # Normalise current dataframe
    foc_data.append(norm_df) # Store normalised dataframe into list

# Loop through all STRESSED dataframes
for df in str_data_raw:
    norm_df = normalise_data(df) # Normalise current dataframe
    str_data.append(norm_df) # Store normalised dataframe into list

# Obtain training, validation and test splits
base_train_data, base_val_data, base_test_data = split_data(base_data, 0.8, 0.19, 0.01)
dist_train_data, dist_val_data, dist_test_data = split_data(dist_data, 0.8, 0.19, 0.01)
foc_train_data, foc_val_data, foc_test_data = split_data(foc_data, 0.8, 0.19, 0.01)
str_train_data, str_val_data, str_test_data = split_data(str_data, 0.5, 0.2, 0.3)
# print(base_train_data)

# Obtain labels for training, validation and test splits
base_train_labels, base_val_labels, base_test_labels = create_labels([base_train_data, base_val_data, base_test_data], 0)
dist_train_labels, dist_val_labels, dist_test_labels = create_labels([dist_train_data, dist_val_data, dist_test_data], 1)
foc_train_labels, foc_val_labels, foc_test_labels = create_labels([foc_train_data, foc_val_data, foc_test_data], 2)
str_train_labels, str_val_labels, str_test_labels = create_labels([str_train_data, str_val_data, str_test_data], 3)
# print(str_test_labels)

# Place all training, validation and testing data and labels into lists
train_data_list = [base_train_data, dist_train_data, foc_train_data, str_train_data]
train_labels_list = [base_train_labels, dist_train_labels, foc_train_labels, str_train_labels]

val_data_list = [base_val_data, dist_val_data, foc_val_data, str_val_data]
val_labels_list = [base_val_labels, dist_val_labels, foc_val_labels, str_val_labels]

test_data_list = [base_test_data, dist_test_data, foc_test_data, str_test_data]
test_labels_list = [base_test_labels, dist_test_labels, foc_test_labels, str_test_labels]

# Window the training, validation and test splits
train_data, train_labels = window_data(train_data_list, train_labels_list, window_size, window_overlap)
val_data, val_labels = window_data(val_data_list, val_labels_list, window_size, window_overlap)
test_data, test_labels = window_data(test_data_list, test_labels_list, window_size, window_overlap)
print(train_data.shape)


# ### DEBUGGING
# # print(train_data_raw.head()) # Print first 5 rows of data for inspection
# # print(test_data_raw.head()) # Print first 5 rows of data for inspection
# # print(test_data.shape)
# # plt.plot(train_data_np[:,2])

(13639, 16, 20)


In [196]:
##### Create dataloaders #####
# Convert data to tensors
train_inputs_tensor = torch.tensor(train_data, dtype=torch.float32)
train_labels_tensor = torch.tensor(train_labels, dtype=torch.long)#.unsqueeze(1) # nn.CrossEntropyLoss() expects integer class labels, no floats or one-hot. Also no need for unsqueeze(1) for CrossEntropyLoss, it just take labels with dim [batch size]

val_inputs_tensor = torch.tensor(val_data, dtype=torch.float32)
val_labels_tensor = torch.tensor(val_labels, dtype=torch.long)#.unsqueeze(1)

test_inputs_tensor = torch.tensor(test_data, dtype=torch.float32)
test_labels_tensor = torch.tensor(test_labels, dtype=torch.long)#.unsqueeze(1)

# Build data loaders
train_dataset = TensorDataset(train_inputs_tensor, train_labels_tensor)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

val_dataset = TensorDataset(val_inputs_tensor, val_labels_tensor)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)#True)

test_dataset = TensorDataset(test_inputs_tensor, test_labels_tensor)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)#True)


## DEBUGGING
print(f"Shape of training inputs: {train_inputs_tensor.shape}")
print(f"Shape of training labels: {train_labels_tensor.shape}")

print(f"Shape of validation inputs: {val_inputs_tensor.shape}")
print(f"Shape of validation labels: {val_labels_tensor.shape}")

print(f"Shape of testing inputs: {test_inputs_tensor.shape}")
print(f"Shape of testing labels: {test_labels_tensor.shape}")

print(f"Train labels min: {train_labels_tensor.min()}, max: {train_labels_tensor.max()}")
print(f"Train labels shape: {train_labels_tensor.shape}")

Shape of training inputs: torch.Size([13639, 16, 20])
Shape of training labels: torch.Size([13639])
Shape of validation inputs: torch.Size([3621, 16, 20])
Shape of validation labels: torch.Size([3621])
Shape of testing inputs: torch.Size([1560, 16, 20])
Shape of testing labels: torch.Size([1560])
Train labels min: 0, max: 3
Train labels shape: torch.Size([13639])


In [197]:
##### Model definition #####
class LSTMClassifier(nn.Module):
    def __init__(self, input_size=num_sensor_readings, hidden_size=16, output_size=num_classes): # Note: output_size should be equal to number of classification classes
        super().__init__()
        # self.lstm1 = nn.LSTM(input_size=input_size, hidden_size=hidden_size, batch_first=True, dropout=0.2) # Input: [batch, sequence length, input dimension (number of sensors)]
        self.lstm1 = nn.LSTM(input_size=input_size, hidden_size=hidden_size, num_layers=6, batch_first=True, dropout=0.3) # Input: [batch, sequence length, input dimension (number of sensors)]
        # self.relu1 = nn.ReLU() # ReLU
        self.dropout1 = nn.Dropout(0.3) # Dropout

        # self.lstm2 = nn.LSTM(input_size=hidden_size, hidden_size=hidden_size, batch_first=True, dropout=0.2) # Input: [batch, sequence length, input dimension (number of sensors)]
        # self.relu2 = nn.ReLU() # ReLU
        # self.dropout2 = nn.Dropout(0.2) # Dropout

        # More LSTM 3,5, no dropouts

        self.fc1 = nn.Linear(hidden_size, hidden_size)
        self.layernorm1 = nn.LayerNorm(hidden_size) # Layer norm
        self.relu3 = nn.ReLU() # ReLU
        self.dropout3 = nn.Dropout(0.3) # Dropout

        self.fc2 = nn.Linear(hidden_size, output_size)
        # self.layernorm2 = nn.LayerNorm(output_size) # Layer norm
        # self.relu4 = nn.ReLU() # ReLU
        # self.dropout4 = nn.Dropout(0.3) # Dropout
    
    def forward(self, x):
        out, _ = self.lstm1(x) # Output: [batch, sequence length, hidden dimension (number of sensors)]
        # out = out[:, -1, :] # Use final hidden state of model as the output classification (Output: [batch, hidden dimension])
        # out = self.relu1(out)
        # out = self.dropout1(out)
        
        # out, _ = self.lstm2(out)
        # out = self.relu2(out)
        # out = self.dropout2(out)
        out = out[:, -1, :]
        out = self.dropout1(out)
        
        out = self.fc1(out) # CLASSIFY: Output here are LOGITS (Output: [batch, num_classes])
        out = self.layernorm1(out)
        out = self.relu3(out)
        out = self.dropout3(out)

        out = self.fc2(out) # CLASSIFY: Output here are LOGITS (Output: [batch, num_classes])
        # out = self.layernorm2(out)
        # out = self.relu4(out)
        # out_logits = self.dropout4(out)

        return out # AS LOGITS

In [198]:
##### Traing model #####
model = LSTMClassifier()#.to(DEVICE) # Define model
print(torchinfo.summary(model, input_size=(1, window_size, num_sensor_readings))) # Input: [batch size, sequence length, input size (number of sensors)]

# Train model
train(
    model,
    train_loader,
    nn.CrossEntropyLoss(), #nn.CrossEntropyLoss() for multi-class classification, nn.BCEWithLogitsLoss() for binary classification, nn.MSELoss()
    optim.Adam(model.parameters(), lr=0.001),
    num_epochs=10,#50,#500
    val_loader=val_loader,
    print_loss=True,
)


# Test model
eval(
    model,
    test_loader,
    nn.CrossEntropyLoss(), #nn.CrossEntropyLoss() for multi-class classification, nn.BCEWithLogitsLoss() for binary classification, nn.MSELoss()
    print_loss=True,
)

Layer (type:depth-idx)                   Output Shape              Param #
LSTMClassifier                           [1, 4]                    --
├─LSTM: 1-1                              [1, 16, 16]               13,312
├─Dropout: 1-2                           [1, 16]                   --
├─Linear: 1-3                            [1, 16]                   272
├─LayerNorm: 1-4                         [1, 16]                   32
├─ReLU: 1-5                              [1, 16]                   --
├─Dropout: 1-6                           [1, 16]                   --
├─Linear: 1-7                            [1, 4]                    68
Total params: 13,684
Trainable params: 13,684
Non-trainable params: 0
Total mult-adds (Units.MEGABYTES): 0.21
Input size (MB): 0.00
Forward/backward pass size (MB): 0.00
Params size (MB): 0.05
Estimated Total Size (MB): 0.06


Validation confusion matrix:
 [[280 515  69   0]
 [665 202  40   0]
 [530 281  92   0]
 [360 496  91   0]]
Label: Baseline, Accuracy: 32.4074%
Label: Distracted, Accuracy: 22.2712%
Label: Focused, Accuracy: 10.1883%
Label: Stressed, Accuracy: 0.0000%
Epoch[1/10] Training Loss: 1.34808, Validation Loss: 1.64766
Validation confusion matrix:
 [[300  84 480   0]
 [386 172 349   0]
 [422 111 370   0]
 [433 136 378   0]]
Label: Baseline, Accuracy: 34.7222%
Label: Distracted, Accuracy: 18.9636%
Label: Focused, Accuracy: 40.9745%
Label: Stressed, Accuracy: 0.0000%
Epoch[2/10] Training Loss: 1.22997, Validation Loss: 1.69008
Validation confusion matrix:
 [[193 355 316   0]
 [108 641 158   0]
 [303 435 165   0]
 [306 416 225   0]]
Label: Baseline, Accuracy: 22.3380%
Label: Distracted, Accuracy: 70.6725%
Label: Focused, Accuracy: 18.2724%
Label: Stressed, Accuracy: 0.0000%
Epoch[3/10] Training Loss: 1.10075, Validation Loss: 1.72131
Validation confusion matrix:
 [[228 213 320 103]
 [ 58 435 263 1